In [1]:
# Cài đặt thư viện cần thiết
# Khắc phục lỗi PyTorch bằng cách dùng phiên bản mới được Colab hỗ trợ
!pip install gensim torch==2.3.0 tensorflow -q

# --- Khắc phục Lỗi 2: Thiếu file dữ liệu ---

# 1. Đường dẫn giải nén
# Giả định file hwu.tar.gz được tải lên thư mục gốc của Colab: /content/hwu.tar.gz
FILE_PATH = "/content/hwu.tar.gz"

# 2. Giải nén dữ liệu cho Phần 2 (Text Classification)
print("Bắt đầu giải nén hwu.tar.gz...")
try:
    # Lệnh giải nén sử dụng đường dẫn tuyệt đối
    !tar -xzvf {FILE_PATH}
    print("Giải nén hwu.tar.gz thành công.")
except Exception as e:
    print(f"LỖI GIẢI NÉN: Vui lòng kiểm tra lại đường dẫn {FILE_PATH} và đảm bảo file tồn tại.")
    # Tạo các file rỗng để tránh lỗi FileNotFoundError ở cell sau nếu có lỗi giải nén
    !touch hwu_train.csv hwu_val.csv hwu_test.csv

# 3. Tạo cấu trúc thư mục cho Phần 3 (POS Tagging)
# Các file .conllu cần nằm trong thư mục này
!mkdir -p data/UD_English-EWT
print("Đảm bảo các file UD-EWT đã được tải lên data/UD_English-EWT/")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [10]:
import pandas as pd

# SỬA LẠI PHẦN LOAD DATA
# 1. Đổi sep='\t' thành sep=',' (vì dữ liệu ngăn cách bằng dấu phẩy)
# 2. Đổi header=None thành header=0 (để nó hiểu dòng đầu tiên là tiêu đề và bỏ qua, không coi là dữ liệu)
# 3. Dùng names=['text', 'intent'] để đổi tên cột 'category' trong file thành 'intent' cho khớp với code bên dưới

train_df = pd.read_csv('hwu/train.csv', sep=',', header=0, names=['text', 'intent'])
val_df = pd.read_csv('hwu/val.csv', sep=',', header=0, names=['text', 'intent'])
test_df = pd.read_csv('hwu/test.csv', sep=',', header=0, names=['text', 'intent'])

# Kiểm tra lại dữ liệu sau khi sửa
print("--- Dữ liệu sau khi sửa ---")
print(train_df.head())

# Kiểm tra xem cột intent có dữ liệu thật chưa
print("\n--- Kiểm tra các nhãn (intent) ---")
print(train_df['intent'].unique()[:5]) # In thử 5 nhãn đầu tiên

--- Dữ liệu sau khi sửa ---
                                                text       intent
0                what alarms do i have set right now  alarm_query
1                    checkout today alarm of meeting  alarm_query
2                              report alarm settings  alarm_query
3  see see for me the alarms that you have set to...  alarm_query
4                       is there an alarm for ten am  alarm_query

--- Kiểm tra các nhãn (intent) ---
['alarm_query' 'alarm_remove' 'alarm_set' 'audio_volume_down'
 'audio_volume_mute']


In [11]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Fit trên toàn bộ nhãn có thể có (nối cả train, val, test để đảm bảo không sót nhãn nào)
label_encoder = LabelEncoder()
all_intents = pd.concat([train_df['intent'], val_df['intent'], test_df['intent']])
label_encoder.fit(all_intents)

# Transform (Chuyển chữ thành số)
y_train = label_encoder.transform(train_df['intent'])
y_val = label_encoder.transform(val_df['intent'])
y_test = label_encoder.transform(test_df['intent'])

# Lấy số lượng lớp (class) để dùng cho lớp Dense cuối cùng
num_classes = len(label_encoder.classes_)
print(f"Số lượng intent: {num_classes}")

Số lượng intent: 64


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

# 1. Tạo Pipeline
model_tfidf = make_pipeline(
    TfidfVectorizer(max_features=5000), # Giới hạn 5000 từ quan trọng nhất
    LogisticRegression(max_iter=1000)   # Tăng max_iter để thuật toán hội tụ
)

# 2. Train
model_tfidf.fit(train_df['text'], y_train)

# 3. Evaluate
y_pred_tfidf = model_tfidf.predict(test_df['text'])
print("--- Task 1: TF-IDF Result ---")
print(classification_report(y_test, y_pred_tfidf, target_names=label_encoder.classes_))

--- Task 1: TF-IDF Result ---
                          precision    recall  f1-score   support

             alarm_query       0.90      0.95      0.92        19
            alarm_remove       1.00      0.73      0.84        11
               alarm_set       0.77      0.89      0.83        19
       audio_volume_down       1.00      0.75      0.86         8
       audio_volume_mute       0.92      0.80      0.86        15
         audio_volume_up       0.93      1.00      0.96        13
          calendar_query       0.45      0.53      0.49        19
         calendar_remove       0.89      0.89      0.89        19
            calendar_set       0.87      0.68      0.76        19
          cooking_recipe       0.59      0.68      0.63        19
        datetime_convert       0.67      0.75      0.71         8
          datetime_query       0.74      0.89      0.81        19
        email_addcontact       0.78      0.88      0.82         8
             email_query       0.83      0.79

In [13]:
from gensim.models import Word2Vec

# Chuẩn bị dữ liệu: Word2Vec cần list các list từ (tokenized sentences)
# Ví dụ: [['tôi', 'thích', 'nhạc'], ['bật', 'đèn']]
X_train_tokens = [text.split() for text in train_df['text']]

# Train Word2Vec
# vector_size=100: Mỗi từ biểu diễn bằng vector 100 chiều
w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=100, window=5, min_count=1, workers=4)

In [14]:
def get_sentence_embedding(sentence, model):
    words = sentence.split()
    word_vectors = []
    for word in words:
        if word in model.wv:
            word_vectors.append(model.wv[word])

    if len(word_vectors) > 0:
        return np.mean(word_vectors, axis=0) # Tính trung bình dọc
    else:
        return np.zeros(model.vector_size) # Nếu câu không có từ nào trong từ điển -> vector 0

# Áp dụng cho tập train/val/test
X_train_w2v = np.array([get_sentence_embedding(text, w2v_model) for text in train_df['text']])
X_val_w2v = np.array([get_sentence_embedding(text, w2v_model) for text in val_df['text']])
X_test_w2v = np.array([get_sentence_embedding(text, w2v_model) for text in test_df['text']])

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model_dense = Sequential([
    Dense(128, activation='relu', input_shape=(100,)), # input là vector 100 chiều
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model_dense.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
import numpy as np
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# 1. Compile (Cấu hình)
model_dense.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Dùng sparse vì y_train là số nguyên (0, 1, 2...)
    metrics=['accuracy']
)

# 2. Train (Huấn luyện) - Đây là bước model thực sự học
print("--- Bắt đầu train Task 2 ---")
history_dense = model_dense.fit(
    X_train_w2v, y_train,
    validation_data=(X_val_w2v, y_val),
    epochs=15,          # Số vòng học
    batch_size=32,
    verbose=1           # verbose=1 để hiện thanh tiến trình
)

# 3. Evaluate (Đánh giá)
print("\n--- Kết quả đánh giá Task 2 ---")
# Dự đoán trên tập test
y_pred_probs = model_dense.predict(X_test_w2v)
y_pred = np.argmax(y_pred_probs, axis=1) # Chuyển từ xác suất sang nhãn (0, 1, 2...)

# In báo cáo chi tiết
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

--- Bắt đầu train Task 2 ---
Epoch 1/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.0219 - loss: 4.1443 - val_accuracy: 0.0623 - val_loss: 4.1006
Epoch 2/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.0427 - loss: 4.0984 - val_accuracy: 0.0567 - val_loss: 4.0334
Epoch 3/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0554 - loss: 4.0104 - val_accuracy: 0.0725 - val_loss: 3.8969
Epoch 4/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0737 - loss: 3.8796 - val_accuracy: 0.0874 - val_loss: 3.7539
Epoch 5/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0870 - loss: 3.7356 - val_accuracy: 0.1245 - val_loss: 3.6394
Epoch 6/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0926 - loss: 3.6315 - val_accuracy: 0.1292 - val_loss: 3.5389
Epoch 7/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.1098 - loss: 3.5489 - val_accuracy: 0.1264 - val_loss: 3.4790
Epoch 8/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.1195 - lo

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Tokenizer
tokenizer = Tokenizer(oov_token="<UNK>") # oov_token để xử lý từ lạ
tokenizer.fit_on_texts(train_df['text']) # Chỉ fit trên tập train!

vocab_size = len(tokenizer.word_index) + 1
print(f"Kích thước từ điển: {vocab_size}")

# 2. Chuyển text sang sequence
train_seq = tokenizer.texts_to_sequences(train_df['text'])
val_seq = tokenizer.texts_to_sequences(val_df['text'])
test_seq = tokenizer.texts_to_sequences(test_df['text'])

# 3. Padding (Đảm bảo độ dài bằng nhau)
MAX_LEN = 50 # Bạn có thể chọn độ dài khác tùy vào dữ liệu
X_train_pad = pad_sequences(train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(val_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

Kích thước từ điển: 4265


In [17]:
embedding_dim = 100 # Phải trùng với vector_size của w2v_model ở trên
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]
    # Nếu từ không có trong w2v, nó sẽ giữ nguyên là vector 0

In [18]:
from tensorflow.keras.layers import Embedding, LSTM

model_lstm_pretrained = Sequential([
    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              weights=[embedding_matrix], # NẠP WEIGHTS VÀO ĐÂY
              input_length=MAX_LEN,
              trainable=False),           # ĐÓNG BĂNG (không học lại embedding)
    LSTM(128, dropout=0.2),
    Dense(num_classes, activation='softmax')
])
# Compile và Train tương tự...

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [22]:
model_lstm_scratch = Sequential([
    Embedding(input_dim=vocab_size,
              output_dim=100,       # Tự học vector 100 chiều
              input_length=MAX_LEN,
              trainable=True),      # CHO PHÉP HỌC (Mặc định là True)
    LSTM(128, dropout=0.2),
    Dense(num_classes, activation='softmax')
])
from tensorflow.keras.callbacks import EarlyStopping

# 1. Compile
model_lstm_pretrained.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Thiết lập EarlyStopping để tự dừng nếu không học được nữa (tránh tốn thời gian)
callback = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# 2. Train
print("--- Bắt đầu train Task 3 ---")
history_lstm = model_lstm_pretrained.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[callback],
    verbose=1
)

# 3. Evaluate
print("\n--- Kết quả đánh giá Task 3 ---")
y_pred_probs_lstm = model_lstm_pretrained.predict(X_test_pad)
y_pred_lstm = np.argmax(y_pred_probs_lstm, axis=1)

print(classification_report(y_test, y_pred_lstm, target_names=label_encoder.classes_))

--- Bắt đầu train Task 3 ---
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


280/280 ━━━━━━━━━━━━━━━━━━━━ 29s 79ms/step - accuracy: 0.0160 - loss: 4.1422 - val_accuracy: 0.0344 - val_loss: 4.0178
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 22s 79ms/step - accuracy: 0.0325 - loss: 4.0382 - val_accuracy: 0.0390 - val_loss: 3.9246
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 22s 79ms/step - accuracy: 0.0482 - loss: 3.9172 - val_accuracy: 0.0446 - val_loss: 3.8648
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 21s 74ms/step - accuracy: 0.0444 - loss: 3.9242 - val_accuracy: 0.0586 - val_loss: 3.8289
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 23s 82ms/step - accuracy: 0.0540 - loss: 3.8637 - val_accuracy: 0.0604 - val_loss: 3.7612
Epoch 6/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 23s 81ms/step - accuracy: 0.0530 - loss: 3.8314 - val_accuracy: 0.0743 - val_loss: 3.7559
Epoch 7/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 41s 80ms/step - accuracy: 0.0579 - loss: 3.8082 - val_accuracy: 0.0790 - val_loss: 3.7286
Epoch 8/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 22s 79ms/step - accuracy: 0.0545 - loss: 3.7982 - val_accurac

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Định nghĩa lại tham số (đảm bảo nhất quán)
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100
max_len = 50 # Hoặc số bạn đã chọn ở bước padding

# --- XÂY DỰNG MODEL TASK 4 ---
model_lstm_scratch = Sequential([
    # Điểm khác biệt duy nhất: KHÔNG truyền weights, và để trainable=True (mặc định)
    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              input_length=max_len),

    LSTM(128, dropout=0.2, recurrent_dropout=0.0), # recurrent_dropout=0 để chạy nhanh hơn trên GPU/CPU thường
    Dense(num_classes, activation='softmax')
])

model_lstm_scratch.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# --- TRAIN ---
callback = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("--- Bắt đầu train Task 4 (End-to-End) ---")
history_scratch = model_lstm_scratch.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[callback],
    verbose=1
)

# --- ĐÁNH GIÁ ---
print("\n--- Kết quả Task 4 ---")
y_pred_probs_scratch = model_lstm_scratch.predict(X_test_pad)
y_pred_scratch = np.argmax(y_pred_probs_scratch, axis=1)

print(classification_report(y_test, y_pred_scratch, target_names=label_encoder.classes_))

--- Bắt đầu train Task 4 (End-to-End) ---
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


280/280 ━━━━━━━━━━━━━━━━━━━━ 36s 92ms/step - accuracy: 0.0118 - loss: 4.1466 - val_accuracy: 0.0177 - val_loss: 4.1280
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 26s 93ms/step - accuracy: 0.0213 - loss: 4.1327 - val_accuracy: 0.0177 - val_loss: 4.1300
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 27s 96ms/step - accuracy: 0.0175 - loss: 4.1363 - val_accuracy: 0.0177 - val_loss: 4.1254
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 26s 92ms/step - accuracy: 0.0154 - loss: 4.1318 - val_accuracy: 0.0177 - val_loss: 4.1254
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 26s 91ms/step - accuracy: 0.0213 - loss: 4.1335 - val_accuracy: 0.0177 - val_loss: 4.1260
Epoch 6/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 26s 91ms/step - accuracy: 0.0145 - loss: 4.1361 - val_accuracy: 0.0177 - val_loss: 4.1244
Epoch 7/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 26s 93ms/step - accuracy: 0.0170 - loss: 4.1344 - val_accuracy: 0.0177 - val_loss: 4.1243
Epoch 8/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 26s 91ms/step - accuracy: 0.0143 - loss: 4.1308 - val_accurac

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [26]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report

# --- BƯỚC 1: LOAD DỮ LIỆU LẠI TỪ ĐẦU ---
# Đảm bảo dùng sep=',' như đã sửa
train_df = pd.read_csv('hwu/train.csv', sep=',', header=0, names=['text', 'intent'])
val_df = pd.read_csv('hwu/val.csv', sep=',', header=0, names=['text', 'intent'])
test_df = pd.read_csv('hwu/test.csv', sep=',', header=0, names=['text', 'intent'])

# --- BƯỚC 2: MÃ HÓA NHÃN (LABEL ENCODE) ---
label_encoder = LabelEncoder()
all_intents = pd.concat([train_df['intent'], val_df['intent'], test_df['intent']])
label_encoder.fit(all_intents)

y_train = label_encoder.transform(train_df['intent'])
y_val = label_encoder.transform(val_df['intent'])
y_test = label_encoder.transform(test_df['intent'])
num_classes = len(label_encoder.classes_)

# --- BƯỚC 3: TOKENIZER & PADDING (QUAN TRỌNG NHẤT) ---
tokenizer = Tokenizer(oov_token="<UNK>")
tokenizer.fit_on_texts(train_df['text']) # Fit lại trên dữ liệu mới

vocab_size = len(tokenizer.word_index) + 1
max_len = 50

# Chuyển text sang sequence
X_train_seq = tokenizer.texts_to_sequences(train_df['text'])
X_val_seq = tokenizer.texts_to_sequences(val_df['text'])
X_test_seq = tokenizer.texts_to_sequences(test_df['text'])

# Padding
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

# ===> KIỂM TRA DỮ LIỆU (DEBUG) <===
print("--- Kiểm tra mẫu dữ liệu ---")
print(f"Câu gốc: {train_df.iloc[0]['text']}")
print(f"Vector số: {X_train_pad[0]}")
print(f"Vocab size: {vocab_size}")
if np.max(X_train_pad) == 0:
    print("CẢNH BÁO: Dữ liệu toàn số 0! Tokenizer bị lỗi.")
else:
    print("Dữ liệu có vẻ ổn.")
from tensorflow.keras.optimizers import Adam

# --- BƯỚC 4: XÂY DỰNG MODEL (ĐÃ SỬA) ---
model_scratch = Sequential([
    # THÊM mask_zero=True ĐỂ BỎ QUA PADDING
    Embedding(input_dim=vocab_size,
              output_dim=100,
              mask_zero=True),  # <--- DÒNG QUAN TRỌNG NHẤT

    LSTM(128, dropout=0.3, recurrent_dropout=0.0),
    Dense(num_classes, activation='softmax')
])

# Dùng Learning Rate lớn hơn một chút (0.005) để thoát khỏi điểm chết
optimizer = Adam(learning_rate=0.005)

model_scratch.compile(optimizer=optimizer,
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

# --- BƯỚC 5: TRAIN ---
callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("\n--- Bắt đầu Train Task 4 (Fix Padding + Masking) ---")
history = model_scratch.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=25,
    batch_size=32,
    callbacks=[callback],
    verbose=1
)

# --- BƯỚC 6: ĐÁNH GIÁ ---
print("\n--- Kết quả Task 4 ---")
y_pred = np.argmax(model_scratch.predict(X_test_pad), axis=1)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

--- Kiểm tra mẫu dữ liệu ---
Câu gốc: what alarms do i have set right now
Vector số: [ 9 99 24  5 26 35 92 62  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0]
Vocab size: 4265
Dữ liệu có vẻ ổn.

--- Bắt đầu Train Task 4 (Fix Padding + Masking) ---
Epoch 1/25
280/280 ━━━━━━━━━━━━━━━━━━━━ 43s 116ms/step - accuracy: 0.3114 - loss: 2.8964 - val_accuracy: 0.8104 - val_loss: 0.7355
Epoch 2/25
280/280 ━━━━━━━━━━━━━━━━━━━━ 29s 103ms/step - accuracy: 0.8553 - loss: 0.5667 - val_accuracy: 0.8485 - val_loss: 0.5532
Epoch 3/25
280/280 ━━━━━━━━━━━━━━━━━━━━ 29s 104ms/step - accuracy: 0.9432 - loss: 0.2198 - val_accuracy: 0.8643 - val_loss: 0.5223
Epoch 4/25
280/280 ━━━━━━━━━━━━━━━━━━━━ 43s 112ms/step - accuracy: 0.9714 - loss: 0.1081 - val_accuracy: 0.8550 - val_loss: 0.5499
Epoch 5/25
280/280 ━━━━━━━━━━━━━━━━━━━━ 40s 108ms/step - accuracy: 0.9796 - loss: 0.0712 - val_accuracy: 0.8597 - val_loss: 0.5855
Epoch 6/25
280/28